# Gun 2 - Embedding Uretimi

Metin bloklarini yerel, cok dilli bir sentence-transformers modeliyle vektore ceviriyoruz.

In [1]:
import sys
import os
import json
import time

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from text_splitter import split_text
from embedder import embed_chunks, embedding_dimension, cosine_similarity

print("text_splitter ve embedder yuklendi.")

text_splitter ve embedder yuklendi.


## 1. Temel davranis testi

In [2]:
sample_chunks = [
    {"chunk_id": 0, "text": "Merhaba dunya.", "token_count": 4},
    {"chunk_id": 1, "text": "Bu bir test metni.", "token_count": 5},
]

embedded_sample = embed_chunks(sample_chunks)
dim = embedding_dimension()

assert len(embedded_sample) == len(sample_chunks)
for original, emb in zip(sample_chunks, embedded_sample):
    assert emb["chunk_id"] == original["chunk_id"]
    assert emb["text"] == original["text"]
    assert emb["token_count"] == original["token_count"]
    assert len(emb["embedding"]) == dim

print(f"OK - {len(embedded_sample)} chunk embedlendi, boyut: {dim}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

OK - 2 chunk embedlendi, boyut: 384


In [3]:
# bos liste icin de guvenli calismali
assert embed_chunks([]) == []
print("OK - bos liste icin bos liste donuyor")

OK - bos liste icin bos liste donuyor


## 2. Gercek belge verisiyle uctan uca test

In [4]:
GT_PATH = "../data/processed/ground_truth.json"
with open(GT_PATH, encoding="utf-8") as f:
    ground_truth = json.load(f)


def format_document(fields: dict) -> str:
    return (
        f"Talep Eden: {fields['talep_eden']}\n"
        f"Tarih: {fields['tarih']}\n"
        f"Departman: {fields['departman']}\n"
        f"Konu: {fields['konu']}\n\n"
        f"{fields['aciklama']}"
    )


documents = [format_document(v) for _, v in sorted(ground_truth.items())]
corpus = "\n\n---\n\n".join(documents)

chunks = split_text(corpus, chunk_size=60, chunk_overlap=15)
print(f"{len(chunks)} chunk uretildi.")

11 chunk uretildi.


In [5]:
t0 = time.time()
embedded_chunks = embed_chunks(chunks)
elapsed = time.time() - t0

print(f"{len(embedded_chunks)} chunk embedlendi, {elapsed:.2f}s surdu.")
print(f"Embedding boyutu: {embedding_dimension()}")
print(f"Ornek (chunk 0) ilk 8 deger: {embedded_chunks[0]['embedding'][:8]}")

11 chunk embedlendi, 0.08s surdu.
Embedding boyutu: 384
Ornek (chunk 0) ilk 8 deger: [-0.10432437807321548, -0.020960472524166107, 0.008707263506948948, -0.04567227512598038, -0.014642973430454731, 0.009098256938159466, -0.08820517361164093, 0.02144530601799488]


## 3. Anlamsal benzerlik kontrolu

In [6]:
monitor_chunk = embedded_chunks[0]  # monitor talebiyle basliyor
keyboard_chunk = next(c for c in embedded_chunks if "Klavye" in c["text"])
screen_chunk = next(c for c in embedded_chunks if "Ekran" in c["text"])

sim_related = cosine_similarity(monitor_chunk["embedding"], screen_chunk["embedding"])
sim_unrelated = cosine_similarity(monitor_chunk["embedding"], keyboard_chunk["embedding"])

print(f"monitor <-> ek ekran benzerligi : {sim_related:.4f}")
print(f"monitor <-> klavye benzerligi    : {sim_unrelated:.4f}")

assert sim_related > sim_unrelated
print("OK - konu bakimindan yakin belgeler daha yuksek benzerlik skoru aldi")

monitor <-> ek ekran benzerligi : 0.6282
monitor <-> klavye benzerligi    : 0.5461
OK - konu bakimindan yakin belgeler daha yuksek benzerlik skoru aldi


## 4. Sonuclarin kaydedilmesi

In [7]:
OUT_PATH = "../data/processed/chunk_embeddings.json"
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(embedded_chunks, f, ensure_ascii=False, indent=2)

print(f"{len(embedded_chunks)} embedding '{OUT_PATH}' dosyasina kaydedildi.")

11 embedding '../data/processed/chunk_embeddings.json' dosyasina kaydedildi.
